# Launching the bench step by step, up to a call on the Calypso DSP

This notebook starts the bench **one process at a time** and checks each one before adding the next:
first the Osmocom network (STP, HLR, MGW, MSC, BSC, SIP side, BTS), then the emulated phone (the C54x DSP,
QEMU with the ARM and the layer 1 firmware, osmocon, mobile, the bridge), and finally camp, location
update and a call. It runs the same commands as `start-native.sh` and `c54x_exe/run.sh`, written out by
hand, so you can see what each component does and where a fault comes from.

**It installs nothing.** Run it where the bench is already built: the `osmocom-run` image from
`build.sh`, the ISO, or a machine after the [install notebook](installer-osmo-jusquau-dsp.ipynb).

| | |
|---|---|
| Needs | everything under `/opt/GSM` (osmo-operator, c54x_exe, qosmo, osmocom-bb, firmware, DSP ROM) and the Osmocom daemons in the `PATH` |
| Runs as | root (or a kernel started after `sudo -v`) |
| Logs | `/tmp/c54x-pont/<name>.log`, one per process, pid in `<name>.pid` |
| Time | a few minutes |

Rules: one process = one cell, followed by its **check**. A check exits with an error if what it waits
for does not show up, and prints the end of the log. Do not move on after a failed check: fix it first,
the next steps depend on it. Stop everything with the last cell before starting over.

In the `osmocom-run` image:

```bash
docker run -it --rm --privileged --shm-size=2g -p 8888:8888 --entrypoint bash osmocom-run:latest
python3 -m venv /venv && /venv/bin/pip install -q jupyterlab
cd /opt/GSM/osmo-operator/wiki && /venv/bin/jupyter lab --ip=0.0.0.0 --allow-root --no-browser \
  --LabApp.default_url=/lab/tree/lancer-pas-a-pas.ipynb
```

## 0. Helpers, cleanup, configs

The helpers go into `/tmp/c54x-pont/lib.sh`, sourced by every cell (`%%bash` cells do not share their
environment):

- `start NAME CMD…` starts a process detached from the notebook, log in `NAME.log`, pid in `NAME.pid`;
- `wait_log FILE PATTERN [SECONDS]` waits for a line, prints it, or prints the end of the log and fails;
- `wait_tcp PORT [SECONDS]` / `wait_udp PORT [SECONDS]` wait for a listening port;
- `vty PORT COMMAND` sends one command to an Osmocom VTY.

The three `CALYPSO_*` variables are the DSP-mode settings `start-native.sh` exports before handing over
to `c54x_exe/run.sh`; every phone-side process inherits them.

In [ ]:
%%bash
mkdir -p /tmp/c54x-pont
cat > /tmp/c54x-pont/lib.sh <<'EOF'
L=/tmp/c54x-pont
ROOT=/opt/GSM
O=$ROOT/osmo-operator
export CALYPSO_BSP_STREAM=1        # one TS0 burst per frame, in FN order
export CALYPSO_RHEA_DMA_XFER=1     # without it: no SB, no BCCH, no SI
export CALYPSO_PONT_LOCKSTEP=1     # QEMU only advances the frame once the DSP is done
[ "$(id -u)" = 0 ] && SUDO="" || SUDO="sudo -n"

start() {   # start NAME CMD...
  local n=$1; shift
  setsid nohup "$@" > "$L/$n.log" 2>&1 < /dev/null &
  echo $! > "$L/$n.pid"
  echo "started $n (pid $!)"
}
alive() { kill -0 "$(cat "$L/$1.pid" 2>/dev/null)" 2>/dev/null; }
wait_log() {   # wait_log FILE PATTERN [SECONDS]
  local f=$1 p=$2 t=${3:-10} i
  for i in $(seq $((t*2))); do
    grep -aq -- "$p" "$f" 2>/dev/null && { echo "ok      $(grep -a -- "$p" "$f" | tail -1)"; return 0; }
    sleep 0.5
  done
  echo "TIMEOUT waiting for \"$p\" in $f"; tail -15 "$f" 2>/dev/null; return 1
}
wait_tcp() { local i; for i in $(seq $((${2:-10}*2))); do ss -ltnH "( sport = :$1 )" | grep -q . && { echo "ok      tcp $1 listening"; return 0; }; sleep 0.5; done; echo "TIMEOUT tcp $1"; return 1; }
wait_udp() { local i; for i in $(seq $((${2:-10}*2))); do ss -ulnH "( sport = :$1 )" | grep -q . && { echo "ok      udp $1 bound"; return 0; }; sleep 0.5; done; echo "TIMEOUT udp $1"; return 1; }
vty() { { printf 'enable\n%s\n' "$2"; sleep 1; } | timeout --foreground 6 nc -N 127.0.0.1 "$1" 2>/dev/null | tr -d '\r'; }
EOF
echo "helpers written to /tmp/c54x-pont/lib.sh"

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: the pieces this notebook runs, and nothing from a previous session still running
fail=0
for f in "$ROOT/c54x_exe/c54x_exe" "$ROOT/c54x_exe/mobile_pont.cfg" "$ROOT/qosmo/build/qemu-system-arm" \
         "$ROOT/osmocom-bb/src/host/osmocon/osmocon" "$ROOT/firmware/board/compal_e88/layer1.highram.elf" \
         "$ROOT/firmware/board/compal_e88/layer1.highram.bin" "$O/pont/pont_dsp.py"; do
  [ -e "$f" ] && echo "ok      $f" || { echo "MISSING $f"; fail=1; }
done
for b in osmo-stp osmo-hlr osmo-mgw osmo-msc osmo-bsc osmo-bts-trx osmo-sip-connector asterisk mobile nc socat; do
  command -v "$b" >/dev/null && echo "ok      $b" || { echo "MISSING $b"; fail=1; }
done
running=$(pgrep -a -f 'osmo-(stp|hlr|mgw|msc|bsc|bts-trx|sip-connector)|c54x_exe|qemu-system-arm|osmocon|mobile -c|pont_dsp' || true)
[ -n "$running" ] && { echo "ALREADY RUNNING (run the Stop cell at the end first):"; echo "$running"; fail=1; }
exit $fail

The Osmocom configs in `/etc/osmocom` are templates: markers such as `__MCC__`, `__ARFCN__`, `__IMSI__`
are filled from `globals.conf` by `start-native.sh --regen`. A daemon started on an unfilled config
refuses it, so check first.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: no template marker left in the configs this notebook uses
left=$(grep -l '__[A-Z0-9_]*__' /etc/osmocom/osmo-{stp,hlr,mgw,msc,bsc,bts-trx,sip-connector}.cfg \
       "$ROOT/c54x_exe/mobile_pont.cfg" 2>/dev/null || true)
if [ -n "$left" ]; then
  echo "unfilled templates:"; echo "$left"
  echo "fill them first: cd $O && ./start-native.sh --regen, then ./start-native.sh --stop"
  exit 1
fi
echo "ok      all configs filled"
grep -E '^(MCC|MNC|BAND|ARFCN|BSIC|LAC)=' "$O/globals.conf"

---
# Part 1: the network

Bench order: STP, HLR, MGW, MSC, BSC, then the SIP side for calls, then the BTS. Each daemon gets its VTY
checked before the next one starts. VTY ports are the Osmocom defaults: STP 4239, HLR 4258, MGW 4243,
MSC 4254, BSC 4242, SIP connector 4255, BTS 4241.

## 1. osmo-stp, the SS7 signalling point

Every link of the core (BSC↔MSC over SCCP/M3UA) goes through it, so it starts first.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
start stp osmo-stp -c /etc/osmocom/osmo-stp.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check
wait_tcp 4239 && vty 4239 "show cs7 instance 0 asp"

## 2. osmo-hlr, the subscriber database

The database path is set in `osmo-hlr.cfg` (`/var/lib/osmocom/hlr.db`).

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
mkdir -p /var/lib/osmocom
start hlr osmo-hlr -c /etc/osmocom/osmo-hlr.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check
wait_tcp 4258 && vty 4258 "show subscribers all" | tail -n +3

### 2b. The phone's subscriber

The emulated phone uses the test SIM of `mobile_pont.cfg` (`test-sim`: `imsi`, `ki`). Its IMSI must be in
the HLR with the same Ki, or the location update is rejected. This cell reads them from the config and
creates the subscriber if it is missing. The MSISDN is up to you.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
cfg="$ROOT/c54x_exe/mobile_pont.cfg"
IMSI=$(awk '/^ *imsi [0-9]+/ {print $2; exit}' "$cfg")
ALGO=$(awk '/^ *ki / {print $2; exit}' "$cfg")
KI=$(awk '/^ *ki / {s=""; for (i=3; i<=NF; i++) s=s $i; print s; exit}' "$cfg")
MSISDN=1001
echo "IMSI=$IMSI ALGO=$ALGO KI=$KI"
[ -n "$IMSI" ] && [ -n "$KI" ] || { echo "no test-sim imsi/ki found in $cfg"; exit 1; }
case "$ALGO" in comp128) HALGO=comp128v1 ;; xor) HALGO=xor-2g ;; *) HALGO=$ALGO ;; esac
if vty 4258 "show subscriber imsi $IMSI" | grep -q "IMSI: $IMSI"; then
  echo "subscriber already present"
else
  { printf 'enable\n'
    printf 'subscriber imsi %s create\n' "$IMSI"
    printf 'subscriber imsi %s update msisdn %s\n' "$IMSI" "$MSISDN"
    printf 'subscriber imsi %s update aud2g %s ki %s\n' "$IMSI" "$HALGO" "$KI"
    sleep 1; } | timeout 6 nc -N 127.0.0.1 4258 | tr -d '\r' | grep -iE 'error|%' || true
fi

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: the subscriber, with its 2G key
IMSI=$(awk '/^ *imsi [0-9]+/ {print $2; exit}' "$ROOT/c54x_exe/mobile_pont.cfg")
out=$(vty 4258 "show subscriber imsi $IMSI"); echo "$out" | tail -n +3
echo "$out" | grep -q "IMSI: $IMSI" && echo "$out" | grep -qi "2G" || { echo "subscriber or its 2G key missing"; exit 1; }

## 3. osmo-mgw, the media gateway

Carries the voice (RTP) between the BTS and the MSC side.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
start mgw osmo-mgw -c /etc/osmocom/osmo-mgw.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check
wait_tcp 4243 && vty 4243 "show mgcp" | head -8

## 4. osmo-msc, the switching centre

It connects to the STP (SCCP) and to the HLR (GSUP). Its calls go out through an **external MNCC socket**
(`mncc external /tmp/msc_mncc` in `osmo-msc.cfg`): without the SIP connector of step 6, the network
works for camp, LU and SMS, but a call cannot be set up.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
start msc osmo-msc -c /etc/osmocom/osmo-msc.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: VTY, the MNCC socket, and the MSC's SCCP user on the STP
wait_tcp 4254 && test -S /tmp/msc_mncc && echo "ok      /tmp/msc_mncc"
vty 4239 "show cs7 instance 0 asp"

## 5. osmo-bsc, the base station controller

Connects to the MSC through the STP, and waits for the BTS (OML/RSL) on Abis.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
start bsc osmo-bsc -c /etc/osmocom/osmo-bsc.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: VTY, and BTS 0 declared (OML not connected yet: the BTS comes at step 7)
wait_tcp 4242 && vty 4242 "show bts 0" | grep -E "BTS 0|OML Link"

## 6. Asterisk and osmo-sip-connector, for calls

`osmo-sip-connector` takes the MSC's MNCC socket and turns the call into SIP towards Asterisk, where
`extensions.conf` routes number 600 to the echo test. Skip this step if you only want camp, LU and SMS.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
ls /etc/asterisk/extensions.conf /etc/asterisk/pjsip.conf || { echo "Asterisk configs missing"; exit 1; }
mkdir -p /var/run/asterisk /var/log/asterisk /var/lib/asterisk
asterisk -rx "core show version" >/dev/null 2>&1 || asterisk
start sip osmo-sip-connector -c /etc/osmocom/osmo-sip-connector.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: Asterisk answers, number 600 exists, the connector holds the MNCC socket.
# Asterisk forks and needs a few seconds before its control socket answers.
for i in $(seq 1 20); do asterisk -rx "core show version" 2>/dev/null && break; sleep 0.5; done
asterisk -rx "core show version" >/dev/null 2>&1 || { echo "Asterisk not answering after 10 s"; tail -15 /var/log/asterisk/messages* 2>/dev/null; exit 1; }
asterisk -rx "dialplan show 600@default" 2>/dev/null | grep -E "600|Echo" || asterisk -rx "dialplan show" | grep -m3 600
wait_tcp 4255 && grep -ai "mncc" "$L/sip.log" | tail -3

## 7. osmo-bts-trx, the base station

It connects to the BSC and starts sending its TRXC commands to UDP 5701, where **nobody answers yet**:
the bridge (step 12) will. That is expected.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
start bts osmo-bts-trx -c /etc/osmocom/osmo-bts-trx.cfg

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: VTY, the clock/data ports, and OML seen by the BSC
wait_tcp 4241 || exit 1
ss -ulnp | grep -E ':580[0-2] ' || { echo "osmo-bts-trx not bound on 5800-5802"; exit 1; }
sleep 2; vty 4242 "show bts 0" | grep -E "OML Link"

---
# Part 2: the emulated phone

Five processes, in this order: the C54x DSP, QEMU (ARM + layer 1 firmware), osmocon (loads the firmware
over the serial line), mobile (layers 2/3), and the bridge between the BTS and the DSP.

## 8. c54x_exe, the DSP

It runs TI's mask ROM, publishes the shared API RAM (`/dev/shm/calypso_api_ram`) and its socket
(`/tmp/calypso_dsp.sock`), then waits for the ARM. It also listens on UDP 6702 for bursts (BSP).

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
rm -f /dev/shm/calypso_api_ram /tmp/calypso_dsp.sock /dev/shm/calypso_tch_cfg
cd "$ROOT/c54x_exe"
start dsp env CALYPSO_IQDUMP_FCCH=1 CALYPSO_BSP_ATTENTE_MS=40 \
  ./c54x_exe --arm --insns 120000 --iq none --amp 30000 -v

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check ("en attente de l'ARM" = "waiting for the ARM")
wait_log "$L/dsp.log" "en attente de l'ARM" 5 && test -S /tmp/calypso_dsp.sock && echo "ok      /tmp/calypso_dsp.sock" && wait_udp 6702

## 9. QEMU (qosmo): the ARM and the layer 1 firmware

`-M calypso` emulates the Compal E88 board. `CALYPSO_DSP_EXTERN=1` hands the DSP to `c54x_exe` instead of
QEMU's built-in model. The first serial port becomes a pty: that is the modem line osmocon will talk to.
The gdb stub listens on 1234.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
rm -f /tmp/qemu-monitor-pont.sock
start qemu env CALYPSO_PONT_RETRY_DIV=64 CALYPSO_DSP_EXTERN=1 \
  "$ROOT/qosmo/build/qemu-system-arm" -M calypso -cpu arm946 -display none -parallel none \
    -serial pty -serial pty -monitor unix:/tmp/qemu-monitor-pont.sock,server,nowait \
    -gdb tcp:127.0.0.1:1234 \
    -kernel "$ROOT/firmware/board/compal_e88/layer1.highram.elf"

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: the modem pty, the DSP bridge on the QEMU side ("API RAM partagee" = "shared API RAM"), the VM running
wait_log "$L/qemu.log" "label serial0" 10 || exit 1
wait_log "$L/qemu.log" "pont DSP : API RAM partagee" 10 || exit 1
sed -n 's/.*redirected to \(\/dev\/pts\/[0-9]*\) (label serial0).*/\1/p' "$L/qemu.log" | head -1 > "$L/modem.pty"
echo "modem pty: $(cat "$L/modem.pty")"
printf 'info status\n' | socat - UNIX-CONNECT:/tmp/qemu-monitor-pont.sock | grep "VM status"

## 10. osmocon: load the firmware, relay L1CTL

`-m romload` speaks the Calypso boot ROM protocol over the modem pty and uploads `layer1.highram.bin`.
Then osmocon relays layer 1 control messages (L1CTL) on `/tmp/osmocom_l2`, where `mobile` will connect.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
rm -f /tmp/osmocom_l2
start osmocon stdbuf -oL -eL "$ROOT/osmocom-bb/src/host/osmocon/osmocon" -m romload -i 100 \
  -p "$(cat "$L/modem.pty")" -s /tmp/osmocom_l2 \
  "$ROOT/firmware/board/compal_e88/layer1.highram.bin"

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: the upload finished and the L1CTL socket exists; the DSP has booted on the firmware's reset
wait_log "$L/osmocon.log" "your code is running now" 30 || exit 1
test -S /tmp/osmocom_l2 && echo "ok      /tmp/osmocom_l2"
grep -a "RESET_DSP relache" "$L/qemu.log" | tail -1
grep -a "DSP boote" "$L/dsp.log" | tail -1

## 11. mobile: layers 2 and 3

The osmocom-bb `mobile` application with `mobile_pont.cfg`: test SIM, cell lock on the bench ARFCN, VTY.
It immediately asks layer 1 for a power scan and a sync on the cell (`L1CTL_FBSB_REQ`). As long as the
bridge is not running there is no downlink, so `FBSB RESP: result=255` in a loop is **expected** here.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
MS_VTY=$(awk '/^ *bind 127\.0\.0\.1 [0-9]+/ {print $3; exit}' "$ROOT/c54x_exe/mobile_pont.cfg")
echo "mobile VTY port from mobile_pont.cfg: $MS_VTY"; echo "$MS_VTY" > "$L/ms_vty"
ss -ltnH "( sport = :$MS_VTY )" | grep -q . && { echo "port $MS_VTY already taken"; exit 1; }
rm -f /tmp/ms_data
start mobile env L23_SYNC_RETRIES_SELECTION=8 stdbuf -oL mobile -c "$ROOT/c54x_exe/mobile_pont.cfg"

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: still alive, VTY up, layer 1 asked to sync on the cell
sleep 2; alive mobile && echo "ok      mobile alive" || { tail -15 "$L/mobile.log"; exit 1; }
wait_tcp "$(cat "$L/ms_vty")" 5
wait_log "$L/osmocon.log" "L1CTL_FBSB_REQ" 5

## 12. pont_dsp.py: the bridge between the BTS and the DSP

This is the radio link. On one side it speaks TRX to osmo-bts-trx (ports 5700–5702: clock, control,
bursts); on the other it feeds the DSP's burst port (6702). Downlink bursts go to the DSP, uplink bursts
(RACH, SDCCH, TCH) come back to the BTS. `ARFCN`/`BSIC` must match the BSC config (514 / 7 on the bench).
`--no-record` skips the I/Q recording to `/root/record.cfile`; remove it if you want the capture.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
cd "$O"
start pont env PONT_BSC_CFG=/etc/osmocom/osmo-bsc.cfg PONT_ARFCN=514 PONT_BSIC=7 \
  python3 pont/pont_dsp.py --dsp-port 6702 --no-record

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: ports, the BTS clock armed ("horloge armee" = "clock armed"), the BTS's TRXC commands, bursts flowing
wait_log "$L/pont.log" "pont TRX : ports" 10 || exit 1
wait_log "$L/pont.log" "horloge armee" 10
grep -a "CTRL " "$L/pont.log" | head -5
sleep 3; grep -a "STATS fn=" "$L/pont.log" | tail -2

---
## 12b. Variant: the phone as the BTS transceiver (`--osmo-trx`), and a second phone over the air

[2026-10-20] Instead of mobile + bridge (steps 11-12), the emulated Calypso can **be the transceiver of
BTS#0**: QEMU boots `trx.highram.elf` (osmocom-bb `jolly/testing`, `trx` application) and the host
`transceiver` program replaces both `mobile` and `pont_dsp.py`: it talks L1CTL to the firmware and TRXD
(5700-5702) to osmo-bts-trx, `--no-sync` (no cell to listen to, BTS mode right after reset), RX mask
`0x01`, TX mask `0x01` (one TX window at TPU slot 5 = downlink TS0). The DSP extension code of the firmware
turns the ROM's RACH transmit task into FB / SB / dummy / normal bursts, which `c54x_exe` publishes in
`/dev/shm/calypso_tx_rom`.

Everything is driven by `c54x_exe/run.sh` with `OSMO_TRX=1` (alias `banc-trx`). Stop steps 8-12 first:
the DSP, QEMU and osmocon are restarted with the other firmware. Note that osmo-bts-trx **exits** when its
transceiver disappears ("No more clock from transceiver"); `run.sh` relaunches it (step 4b) when BTS#0 is
absent.

Then a **second emulated phone** (`INSTANCE=2`, alias `banc-mobile`: its own `/dev/shm` through a private
mount namespace, `/tmp/c54x-pont2`, L1CTL `/tmp/osmocom_l2_2`, BSP UDP 6712, mobile VTY 4347) camps on it:
`pont/air.py` relays the TRX's published bursts to the mobile's BSP and the mobile's bursts back.

State on 2026-10-20 evening: the second phone camps, sends its RACH, gets its IMMEDIATE ASSIGNMENT on the
SDCCH/4 of TS0 and completes the **location update** (`LOCATION UPDATING ACCEPT`, TMSI allocated). Two
prostheses in `c54x_exe` make up for what the firmware's DSP extension code gets wrong on the emulated core:
the SCH is re-encoded before publishing (`MONTANT_ROM_TX_SB_RECODE`) and the exact bits of each uplink
normal burst are written into the extension's result page (`CALYPSO_BSP_TRX_UL`). The uplink is aligned on
the firmware's clock from the SB it emits (`CALYPSO_BSP_TRX`, `BTS_RX_LAG 3` / delta -2). SMS and calls
over this path are not exercised yet. See `wiki/Reference.md` 2.5 and 2.7.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
for n in pont mobile osmocon gdb qemu dsp; do p=$(cat "$L/$n.pid" 2>/dev/null) && kill "$p" 2>/dev/null && echo "stopped $n"; done
sleep 1; rm -f /dev/shm/calypso_api_ram /tmp/calypso_dsp.sock /tmp/osmocom_l2 /tmp/qemu-monitor-pont.sock "$L"/{pont,mobile,osmocon,gdb,qemu,dsp}.pid
# the TRX chain: DSP, QEMU (trx.highram.elf), osmocon, transceiver ; osmo-bts-trx relaunched if it exited
MODE=dsp OSMO_TRX=1 INSNS=120000 LOCKSTEP=1 IQ=none CALYPSO_BSP_STREAM=1 CALYPSO_RHEA_DMA_XFER=1 bash "$ROOT/c54x_exe/run.sh"

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Check: the firmware in BTS mode, FB and SB one burst in ten each, the BTS clock, no stale bursts
sleep 30
grep -a 'BTS MODE' "$L/osmocon.log" | tail -1                      # BTS MODE: 7 514 {5,...} TX 0..0 RX 0..0
grep -a 'rom-tx\] [0-9]' "$L/dsp.log" | tail -1                    # SB and FB ~ 1/10 of the bursts, NB from the BTS
grep -a 'SB recode' "$L/dsp.log" | tail -1
echo "STALE BURST (must stop growing): $(grep -ac STALE "$L/osmocon.log")"
sed 's/\x1b\[[0-9;]*m//g' /var/log/osmocom/osmo-bts-trx.log /run/user/*/osmo-nitb/logs/bts.log 2>/dev/null | grep -E 'clock (started|skew)' | tail -2

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# The second phone (instance 2) and the air between the two Calypsos
INSTANCE=2 MODE=dsp AIR=1 INSNS=120000 LOCKSTEP=1 IQ=none CALYPSO_BSP_STREAM=1 CALYPSO_RHEA_DMA_XFER=1 bash "$ROOT/c54x_exe/run.sh"
sleep 40
tail -1 /tmp/c54x-pont2/air.log                                      # A->B n (NB SB DUMMY FB) : the TRX's bursts reach the mobile
grep -a 'FB1\|SB' /tmp/c54x-pont2/osmocon.log | grep -av 'FBSB_REQ\|DSP Error' | tail -3   # FB detected, then "=> SB ... BSIC=7"
sed 's/\x1b\[[0-9;]*m//g' /tmp/c54x-pont2/mobile.log | grep -iE 'FBSB RESP|SYSTEM INFO|camped' | tail -3
echo "stop: INSTANCE=2 MODE=dsp bash $ROOT/c54x_exe/run.sh --stop ; MODE=dsp bash $ROOT/c54x_exe/run.sh --stop"

---
# Part 3: the service

## 13. System information, then camp

With the downlink flowing, the phone syncs (FCCH, SCH), reads the system information on the BCCH and
camps on the cell.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
MS_VTY=$(cat "$L/ms_vty")
wait_log "$L/mobile.log" "New SYSTEM INFORMATION" 30 || exit 1
for i in $(seq 1 60); do vty "$MS_VTY" "show ms" | grep -q "C3 camped normally" && { echo "ok      camped after $((i*2)) s"; break; }; sleep 2; done
vty "$MS_VTY" "show ms" | grep -E "camped|MM|IMSI|cell"

## 14. Location update

The phone sends a RACH, gets a dedicated channel (SDCCH), authenticates against the HLR and registers.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
grep -a "\[montant\] RACH" "$L/dsp.log" | tail -1
grep -ao 'UL bursts=[0-9]* tard=[0-9]* rach=[0-9]*' "$L/pont.log" | tail -1
wait_log "$L/mobile.log" "LOCATION UPDATING ACCEPT" 60 || exit 1
sed 's/\x1b\[[0-9;]*m//g' "$L/mobile.log" | grep -aoE 'lai=[0-9]+-[0-9]+-[0-9]+' | tail -1
vty 4254 "show subscriber cache" | grep -E 'IMSI|MSISDN'

## 15. A call to 600 (echo test)

Needs step 6. The call goes mobile → BTS → BSC → MSC → MNCC → osmo-sip-connector → Asterisk, and back.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
MS_VTY=$(cat "$L/ms_vty")
vty "$MS_VTY" "call 1 600" > /dev/null
for i in $(seq 1 15); do
  st="$(vty "$MS_VTY" 'show ms' | sed -n 's/^ *call control state: //p' | head -1)"
  echo "${st:-(no call yet)}"; [ "$st" = ACTIVE ] && break; sleep 2
done
grep -a "\[montant\] TCH" "$L/dsp.log" | tail -2

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
# Hang up
vty "$(cat "$L/ms_vty")" "call 1 hangup" > /dev/null; sleep 2
vty "$(cat "$L/ms_vty")" "show ms" | grep -E "call control state|MM" || echo "no call"

---
## Stop and clean up

Reverse order, then the shared files: a leftover `/tmp/ms_data` makes the next mobile's bind fail
silently, and a leftover `/dev/shm/calypso_horloge` slaves the bridge to a dead session.

In [ ]:
%%bash
. /tmp/c54x-pont/lib.sh
for n in pont mobile osmocon gdb qemu dsp bts sip bsc msc mgw hlr stp; do
  p=$(cat "$L/$n.pid" 2>/dev/null) && kill "$p" 2>/dev/null && echo "stopped $n"
done
asterisk -rx "core stop now" 2>/dev/null || true
sleep 1
rm -f /dev/shm/calypso_api_ram /tmp/calypso_dsp.sock /tmp/osmocom_l2 /tmp/qemu-monitor-pont.sock \
      "$L/modem.pty" /tmp/osmocom_sap /tmp/ms_data /dev/shm/calypso_horloge \
      /dev/shm/calypso_rach /dev/shm/calypso_sdcch_ul /dev/shm/calypso_tch_facch_ul \
      /dev/shm/calypso_tch_sacch_ul /dev/shm/calypso_tch_ul /dev/shm/calypso_tch_cfg "$L"/*.pid
pgrep -a -f 'osmo-(stp|hlr|mgw|msc|bsc|bts-trx|sip-connector)|c54x_exe|qemu-system-arm|osmocon|mobile -c|pont_dsp' \
  && echo "still running (above)" || echo "all stopped"